<a href="https://colab.research.google.com/github/nalmutaweaa/Pretrained-Model/blob/main/Pretrained_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Import Libraries

In [1]:
import pandas as pd
import requests #for webscrabing
from bs4 import BeautifulSoup #scraping

In [8]:
#! pip install transformers

In [2]:
#! pip install datasets

## Movie Review - Sentiment Analysis

In [4]:
from datasets import load_dataset
ds = load_dataset("jahjinx/IMDb_movie_reviews")

In [5]:
df = ds['test'].to_pandas() #convert to df
pos =  df[df['label']==1].sample(50,random_state=42) # take 50 pos rev
neg= df[df['label']==0].sample(50,random_state=42) #take 50 neg rev
rev_df= pd.concat([pos,neg]) #concat
rev_df.head(5)

,text,label
6348,Well it might be a kid's movie...perhaps but i...,1
4525,This is a very well written movie full of susp...,1
6657,DER TODESKING is not one of my favorite Jorg B...,1
4774,"Well, I fear that my review of this special wo...",1
345,As a Czech I am very pleased when I read these...,1


In [5]:
rev_model1 =rev_df.copy()
rev_model2 =rev_df.copy()
rev_model3 =rev_df.copy()

### Model1

In [6]:
#https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english
from transformers import pipeline
pipe1 = pipeline("text-classification", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [7]:
#use the model to predict lables
pred1=[]
score1=[]
for text in rev_df['text']:
  results1=pipe1(text, truncation=True)[0]
  pred1.append(results1['label'])
  score1.append(results1['score'])

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [8]:
#add predictions to df
rev_model1['pred']=pred1
rev_model1['pred_num']=rev_model1['pred'].map({'POSITIVE':1,'NEGATIVE':0})
rev_model1['score']=score1

### Model2

In [9]:
#https://huggingface.co/Facececersek/bedrock-sentiment-tiny
from transformers import pipeline
pipe2 = pipeline("text-classification", model="Facececersek/bedrock-sentiment-tiny")

config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 17.5MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

In [10]:
#use the model to predict lables
pred2=[]
score2=[]
for text in rev_df['text']:
  results2=pipe2(text, truncation=True, max_length=512)[0] #has limitaion of 512
  pred2.append(results2['label'])
  score2.append(results2['score'])

In [11]:
#add predictions to df
rev_model2['pred']=pred2
rev_model2['pred_num']=rev_model2['pred'].map({'POSITIVE':1,'NEGATIVE':0})
rev_model2['score']=score2

### Model3

In [12]:
#https://huggingface.co/JustMon1kAa/bert-tiny-sst2-homework
from transformers import pipeline
pipe3 = pipeline("text-classification", model="JustMon1kAa/bert-tiny-sst2-homework")

config.json:   0%|          | 0.00/763 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 17.5MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [13]:
#use the model to predict lables
pred3=[]
score3=[]
for text in rev_df['text']:
  results3=pipe3(text, truncation=True, max_length=512)[0] #has limitaion token max of 512
  pred3.append(results3['label'])
  score3.append(results3['score'])

In [14]:
#add predictions to df
rev_model3['pred']=pred3
rev_model3['pred_num']=rev_model3['pred'].map({'POSITIVE':1,'NEGATIVE':0})
rev_model3['score']=score3

### Models evaluation

Problem: Perform sentiment analysis to identify whether movie reviews are positive or negative.

Users: Movie critics, streaming platforms, or viewers who want to know if a movie is good or not.

Input: Text-based movie reviews.

Output: A prediction of whether the review is positive or negative.

Success: Measure accuracy by comparing the actual labels with the model's predicted labels.

In [15]:
# Model 1 Accuracy
from sklearn.metrics import accuracy_score
accuracy1 = accuracy_score(rev_model1['label'], rev_model1['pred_num'])
print(accuracy1)

0.87


In [16]:
# Model 2 Accuracy
from sklearn.metrics import accuracy_score
accuracy2 = accuracy_score(rev_model2['label'], rev_model2['pred_num'])
print(accuracy2)

0.8


In [17]:
# Model 1 Accuracy
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
accuracy3 = accuracy_score(rev_model3['label'], rev_model3['pred_num'])
print(accuracy3)

0.85


### Testing

In [21]:
# test with confusing sentences
text= [
    "This movie was amazing!",
    "Terrible movie.",
    "The acting was good, but the story was boring.",
    "I can't decide whether I liked this movie.",
    "What am amzaing movie! people said are they joking?? it was not good at all"]

In [22]:
#model 1
for t in text:
  print(t, pipe1(t, truncation=True))

This movie was amazing! [{'label': 'POSITIVE', 'score': 0.9998800754547119}]
Terrible movie. [{'label': 'NEGATIVE', 'score': 0.9998013377189636}]
The acting was good, but the story was boring. [{'label': 'NEGATIVE', 'score': 0.9973094463348389}]
I can't decide whether I liked this movie. [{'label': 'NEGATIVE', 'score': 0.9993476271629333}]
What am amzaing movie! people said are they joking?? it was not good at all [{'label': 'NEGATIVE', 'score': 0.999760091304779}]


In [23]:
#model 2
for t in text:
  print(t, pipe2(t, truncation=True))

This movie was amazing! [{'label': 'POSITIVE', 'score': 0.9794366955757141}]
Terrible movie. [{'label': 'NEGATIVE', 'score': 0.9729459881782532}]
The acting was good, but the story was boring. [{'label': 'POSITIVE', 'score': 0.5033649206161499}]
I can't decide whether I liked this movie. [{'label': 'NEGATIVE', 'score': 0.9258355498313904}]
What am amzaing movie! people said are they joking?? it was not good at all [{'label': 'NEGATIVE', 'score': 0.9128559231758118}]


In [24]:
#model 3
for t in text:
  print(t, pipe3(t, truncation=True))

This movie was amazing! [{'label': 'POSITIVE', 'score': 0.9035869836807251}]
Terrible movie. [{'label': 'NEGATIVE', 'score': 0.8545757532119751}]
The acting was good, but the story was boring. [{'label': 'POSITIVE', 'score': 0.8832857608795166}]
I can't decide whether I liked this movie. [{'label': 'NEGATIVE', 'score': 0.7495665550231934}]
What am amzaing movie! people said are they joking?? it was not good at all [{'label': 'NEGATIVE', 'score': 0.7984664440155029}]


### Error Analysis

In [27]:
#misscalssifed labels
errors1 = rev_model1[
    rev_model1["label"] != rev_model1["pred_num"]
]
errors2 = rev_model2[
    rev_model2["label"] != rev_model2["pred_num"]
]
errors3 = rev_model3[
    rev_model3["label"] != rev_model3["pred_num"]
]

In [29]:
errors1.shape #  model 1 got 13 out of 100 wrong

(13, 5)

In [30]:
errors2.shape # model 2 got 20 out of 100 wrong

(20, 5)

In [31]:
errors3.shape # model 3 got 15 out of 100 wrong

(15, 5)

In [41]:
#model1:assign a reson manually for 5
errors1[["text", "label", "pred", "score"]].head(5)
reasons = {
    373: "Mixed opinions",
    9433: "Ambiguity",
    5402: "Long text / Mixed opinions",
    6213: "Domain-specific terminology",
    8493: "Mixed opinions"
}
errors1["reason"] = errors1.index.map(reasons)

/tmp/ipykernel_1921/991492353.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors1["reason"] = errors1.index.map(reasons)


In [43]:
#model2:assign a reson manually for 5
errors2[["text", "label", "pred", "score"]].head(5)
reasons = {
    2514: "Mixed context",
    373: "Mixed opinions",
    9433: "Ambiguity",
    6401: "Domain-specific terminology",
    4493: "Context confusion"
}
errors2["reason"] = errors2.index.map(reasons)


/tmp/ipykernel_1921/3928883157.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors2["reason"] = errors2.index.map(reasons)


In [45]:
#model3:assign a reson manually for 5
errors3[["text", "label", "pred", "score"]].head(5)
reasons = {
 2514: "Context confusion",
    373: "Mixed opinions",
    9433: "Ambiguity",
    6401: "Domain-specific language",
    2863: "Mixed opinions"
}
errors3["reason"] = errors3.index.map(reasons)


/tmp/ipykernel_1921/2878497181.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors3["reason"] = errors3.index.map(reasons)


### Model Battle

I would choose Model 1 (DistilBERT) because it achieved the highest accuracy (87%) and made the fewest mistakes (13 out of 100 reviews). And does not have a max lenght limitation.

Although it is larger than Models 2 and 3, it performed better at identifying positive and negative sentiment. I think it is the most suitable model for our movie review sentiment analysis application.

### Domain Shift

The models understand movie review sentimen at a general level. However, IMDb reviews are often longer and contain mixed opinions, which may explain some incorrect predictions.

### What to do?


I would choose DistilBERT Model without fine-tuning for now because it achieved the highest accuracy (87%). However, I think it could be improved by using more evaluation data and handling longer reviews more effectively.

## Zero Shot

In [10]:
# take 30 revs
zero_df=  rev_df.sample(30,random_state=42).copy()

In [8]:
#read the reviews
for i, review in enumerate(zero_df["text"]):
    print(f"{i}: {review}\n")

0: I was attracted to this film by its offbeat, low-key, 'real life' story line. That is, a twenty-something guy flops in the Big Apple and comes back home to live with his parents and even more floppy brother. It just might have worked but there's a problem. And that problem's name is Casey Affleck. <br /><br />Casey Affleck is nearly catatonic in this film. His acting mantra must be "exert as little effort as possible at all times". Or "why speak when you can mutter?" Or maybe "put yourself into a coma as soon as the camera rolls". Lips moving when speaking? Barely. Facial expressions? None. Muscles in face? Atrophied. Something? Nothing. ANYthing? Zip.

1: Jane Eyre_ is one of the greatest novels in the English language and this screenwriter should of read it. I hate it when writers use Spark notes for what a novel is all about. This movie is unbearable to watch if you have read the book.<br /><br />The whole 'red room' is so down played that I wonder why they even bother to put it 

In [33]:
# assign approprate class(total 6)
topics = [
    "Acting",
    "Plot and Story",
    "Cinematography and Visual Effects",
    "Acting",
    "Themes and Message",
    "Acting",
    "Themes and Message",
    "Music and Soundtrack",
    "Plot and Story",
    "Themes and Message",
    "Plot and Story",
    "Acting",
    "Acting",
    "Cinematography and Visual Effects",
    "Themes and Message",
    "Acting",
    "Acting",
    "Plot and Story",
    "Themes and Message",
    "Themes and Message",
    "Acting",
    "Themes and Message",
    "Acting",
    "Plot and Story",
    "Pacing and Editing",
    "Acting",
    "Themes and Message",
    "Acting",
    "Acting",
    "Acting"
]
zero_df["topic"] = topics
zero_df.head()

,text,label,pred,score,topic
9840,"I was attracted to this film by its offbeat, l...",0,Acting & Characters,0.346071,Acting
6274,Jane Eyre_ is one of the greatest novels in th...,0,Story & Writing,0.347335,Plot and Story
1674,Film Noire is a genre that requires a certain ...,0,Overall Experience,0.258721,Cinematography and Visual Effects
7478,This is a truly hilarious film and one that I ...,1,Music & Sound,0.365191,Acting
8308,I learned much from previous viewer reactions....,1,Overall Experience,0.301171,Themes and Message


In [34]:
zero_df.columns

Index(['text', 'label', 'pred', 'score', 'topic'], dtype='object')

In [43]:
### Zero Shot Model
from transformers import pipeline
pipe = pipeline("zero-shot-classification", model="Horizon-Labs/multilingual-zeroshot-small")

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

In [44]:
classes = [ "Acting", "Plot and Story",
    "Cinematography and Visual Effects", "Music and Soundtrack",
    "Themes and Message","Pacing and Editing"]

In [51]:
predictions = []
scores = []

for text in zero_df["text"]:
    result = pipe(text, candidate_labels=classes)
    predictions.append(result["labels"][0])
    scores.append(result["scores"][0])

zero_df["pred"] = predictions
zero_df["score"] = scores

In [52]:
#test the model on first review
result= pipe(zero_df["text"].iloc[0], candidate_labels=classes)
result

{'sequence': 'I was attracted to this film by its offbeat, low-key, \'real life\' story line. That is, a twenty-something guy flops in the Big Apple and comes back home to live with his parents and even more floppy brother. It just might have worked but there\'s a problem. And that problem\'s name is Casey Affleck. <br /><br />Casey Affleck is nearly catatonic in this film. His acting mantra must be "exert as little effort as possible at all times". Or "why speak when you can mutter?" Or maybe "put yourself into a coma as soon as the camera rolls". Lips moving when speaking? Barely. Facial expressions? None. Muscles in face? Atrophied. Something? Nothing. ANYthing? Zip.',
 'labels': ['Acting',
  'Music and Soundtrack',
  'Cinematography and Visual Effects',
  'Themes and Message',
  'Plot and Story',
  'Pacing and Editing'],
 'scores': [0.367549866437912,
  0.18563158810138702,
  0.14771884679794312,
  0.1451091468334198,
  0.09867509454488754,
  0.05531542748212814]}

### Zero Shot Evalutions



In [55]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(zero_df["topic"], zero_df["pred"])
print(accuracy)

0.26666666666666666


### Error Analysis

In [32]:
#check wrong classification manually
errors = zero_df[zero_df["topic"] != zero_df["pred"]]
errors[["text", "topic", "pred", "score"]]

,text,topic,pred,score
1674,Film Noire is a genre that requires a certain ...,Visuals & Production,Overall Experience,0.258721
7478,This is a truly hilarious film and one that I ...,Acting & Characters,Music & Sound,0.365191
8308,I learned much from previous viewer reactions....,Themes & Message,Overall Experience,0.301171
591,"Of the three remakes of this plot, I like them...",Acting & Characters,Overall Experience,0.370372
3769,I saw the movie recently during the Boston Fil...,Themes & Message,Overall Experience,0.357638
2955,It makes sense to me that this film is getting...,Music & Sound,Overall Experience,0.243978
367,I didn't know much about this movie going in- ...,Story & Writing,Overall Experience,0.463667
6348,Well it might be a kid's movie...perhaps but i...,Themes & Message,Overall Experience,0.321876
5019,Sniper gives a true new meaning to war movies....,Story & Writing,Overall Experience,0.253068
7209,Young Mr. Lincoln marks the first film of the ...,Story & Writing,Visuals & Production,0.228242


In [50]:
errors.shape

(22, 5)

## Conclusion

Zero-shot model achieved 26.67% accuracy, correctly classifying 8 out of 30 movie reviews.

I think the accuracy was low because most reviews were long and discussed multiple aspects of a movie rather than one specific topic. The model is also relatively small, and the six categories may not cover every review clearly.

Changing the categories did not improve the overall accuracy. I think using shorter, more topic-specific reviews, clearer labels, or a different pretrained model could improve the results.